# Train model video TỪ ĐẦU (không pretrained, không fine-tune) trên Colab Pro
**Runtime → Change runtime type → GPU (A100 nếu có).**

Quy trình 4 giai đoạn, toàn bộ trọng số khởi tạo ngẫu nhiên:
1. Tiền xử lý video → 128x128, 8 fps (chạy 1 lần, có resume)
2. Pretrain tự giám sát **VideoMAE nhỏ** (không cần nhãn) trên cả ~19.900 video
3. Train **captioning** (video đoạn → câu) trên các video đã có chú thích `timestamp + câu`
4. Suy luận: trượt cửa sổ theo thời gian → danh sách `(start, end, caption)`

**Định dạng chú thích** `annotations.json` (bạn tự tạo, đặt trong thư mục Drive):
```json
{
  "P17": {"duration": 38.8, "segments": [
      {"start": 0.0,  "end": 12.5, "caption": "a person spreads fabric on the ironing board"},
      {"start": 12.5, "end": 38.8, "caption": "the person irons the fabric flat"}]}
}
```
`video_id` trùng tên file video (không có đuôi). Khuyến nghị ≥ 5.000 video có chú thích, câu ngắn 5–12 từ.

In [ ]:
!pip -q install decord tqdm
import os, json, math, random, re, glob, subprocess, shutil, time
from pathlib import Path
from collections import Counter
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from google.colab import drive
drive.mount('/content/drive')
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# ====== CẤU HÌNH (sửa cho đúng Drive của bạn) ======
RAW_DIR   = '/content/drive/MyDrive/videos_raw'        # 19.900 video gốc
PROC_DIR  = '/content/drive/MyDrive/videos_128'        # video đã nén (lưu bền trên Drive)
ANN_PATH  = '/content/drive/MyDrive/annotations.json'  # chú thích timestamp + caption
CKPT_DIR  = '/content/drive/MyDrive/ckpt'
LOCAL_DIR = '/content/videos_128'                      # bản sao trên SSD Colab (đọc nhanh)
os.makedirs(PROC_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)

FPS, SIZE, CROP, T = 8, 128, 112, 16     # 16 frame x 112x112
SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda'

## 1. Tiền xử lý video (resume được, chạy song song)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
EXT = ('.mp4', '.mov', '.mkv', '.avi', '.webm')
raws = sorted(p for p in Path(RAW_DIR).rglob('*') if p.suffix.lower() in EXT)
print('video gốc:', len(raws))

def convert(p):
    out = Path(PROC_DIR) / (p.stem + '.mp4')
    if out.exists() and out.stat().st_size > 1000: return 0
    vf = f"fps={FPS},scale=-2:{SIZE}:flags=bicubic,crop={SIZE}:{SIZE}"
    cmd = ['ffmpeg','-y','-loglevel','error','-i',str(p),'-an','-vf',vf,
           '-c:v','libx264','-preset','veryfast','-crf','28','-g','16',str(out)]
    r = subprocess.run(cmd)
    if r.returncode != 0 and out.exists(): out.unlink()
    return r.returncode

from tqdm.auto import tqdm
with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
    errs = sum(1 for r in tqdm(ex.map(convert, raws), total=len(raws)) if r != 0)
print('lỗi:', errs)

In [ ]:
# Copy sang SSD cục bộ để huấn luyện nhanh (chạy lại mỗi khi Colab cấp phiên mới)
if not os.path.isdir(LOCAL_DIR) or len(os.listdir(LOCAL_DIR)) < 100:
    !cp -r "{PROC_DIR}" "{LOCAL_DIR}"
vids = sorted(glob.glob(LOCAL_DIR + '/*.mp4'))
print('video xử lý:', len(vids))
vid_path = {Path(v).stem: v for v in vids}

# Chia train/val/test THEO VIDEO (ổn định theo hash tên)
import zlib
def split_of(vid):
    h = zlib.crc32(vid.encode()) % 100
    return 'train' if h < 90 else ('val' if h < 95 else 'test')

## 2. Dataset + Model VideoMAE nhỏ (tự viết, khởi tạo ngẫu nhiên)

In [ ]:
from decord import VideoReader, cpu
MEAN = torch.tensor([0.45, 0.45, 0.45]).view(3,1,1,1); STD = torch.tensor([0.225]*3).view(3,1,1,1)

def read_frames(path, idx):
    vr = VideoReader(path, ctx=cpu(0), num_threads=1)
    n = len(vr)
    idx = np.clip(np.asarray(idx), 0, n-1)
    return vr.get_batch(idx.tolist()).asnumpy(), n          # T,H,W,3 uint8

def to_tensor(fr, train):
    x = torch.from_numpy(fr).permute(3,0,1,2).float() / 255.   # 3,T,H,W
    H = x.shape[-1]
    if train:
        i = random.randint(0, H-CROP); j = random.randint(0, H-CROP)
        x = x[..., i:i+CROP, j:j+CROP]
        if random.random() < 0.5: x = x.flip(-1)
        x = x * (0.8 + 0.4*random.random()) + (random.random()-0.5)*0.2   # brightness/contrast nhẹ
    else:
        o = (H-CROP)//2; x = x[..., o:o+CROP, o:o+CROP]
    return (x - MEAN) / STD

class PretrainDS(torch.utils.data.Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, k):
        p = vid_path[self.ids[k]]
        vr = VideoReader(p, ctx=cpu(0), num_threads=1); n = len(vr); del vr
        stride = random.choice([1, 2, 3, 4])
        span = (T-1)*stride + 1
        s = random.randint(0, max(0, n-span))
        fr, _ = read_frames(p, s + np.arange(T)*stride)
        return to_tensor(fr, True)

class Patch3D(nn.Module):
    def __init__(s, D): super().__init__(); s.proj = nn.Conv3d(3, D, (2,16,16), (2,16,16))
    def forward(s, x): return s.proj(x).flatten(2).transpose(1,2)   # B,N,D  (N=8*7*7=392)
NTOK = (T//2) * (CROP//16) * (CROP//16)

def blocks(D, heads, depth):
    layer = nn.TransformerEncoderLayer(D, heads, 4*D, 0.0, activation='gelu', batch_first=True, norm_first=True)
    return nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)

class Encoder(nn.Module):
    def __init__(s, D=384, heads=6, depth=8):
        super().__init__()
        s.D = D; s.patch = Patch3D(D)
        s.pos = nn.Parameter(torch.zeros(1, NTOK, D)); nn.init.trunc_normal_(s.pos, std=.02)
        s.blk = blocks(D, heads, depth); s.norm = nn.LayerNorm(D)
    def forward(s, x, keep=None):
        t = s.patch(x) + s.pos
        if keep is not None: t = torch.gather(t, 1, keep[..., None].expand(-1,-1,s.D))
        return s.norm(s.blk(t))

class VideoMAE(nn.Module):
    def __init__(s, D=384, dd=192, mask_ratio=0.9):
        super().__init__()
        s.enc = Encoder(D); s.mr = mask_ratio; s.dd = dd
        s.proj = nn.Linear(D, dd)
        s.mask_tok = nn.Parameter(torch.zeros(1,1,dd)); nn.init.trunc_normal_(s.mask_tok, std=.02)
        s.pos = nn.Parameter(torch.zeros(1, NTOK, dd)); nn.init.trunc_normal_(s.pos, std=.02)
        s.dec = blocks(dd, 6, 4); s.dnorm = nn.LayerNorm(dd)
        s.head = nn.Linear(dd, 2*16*16*3)
    @staticmethod
    def patchify(x):
        B = x.shape[0]
        x = x.view(B,3,T//2,2,CROP//16,16,CROP//16,16).permute(0,2,4,6,3,5,7,1)
        return x.reshape(B, NTOK, 2*16*16*3)
    def forward(s, x):
        B = x.shape[0]; K = int(NTOK*(1-s.mr))
        shuf = torch.rand(B, NTOK, device=x.device).argsort(1); rest = shuf.argsort(1)
        keep = shuf[:, :K]
        z = s.proj(s.enc(x, keep))
        z = torch.cat([z, s.mask_tok.expand(B, NTOK-K, -1)], 1)
        z = torch.gather(z, 1, rest[..., None].expand(-1,-1,s.dd)) + s.pos
        pred = s.head(s.dnorm(s.dec(z)))
        tgt = s.patchify(x); tgt = (tgt - tgt.mean(-1, keepdim=True)) / (tgt.var(-1, keepdim=True).sqrt() + 1e-6)
        mask = torch.ones(B, NTOK, device=x.device); mask[:, :K] = 0
        mask = torch.gather(mask, 1, rest)
        loss = (((pred.float() - tgt.float())**2).mean(-1) * mask).sum() / mask.sum()
        return loss

m = VideoMAE()
print('tham số encoder: %.1fM' % (sum(p.numel() for p in m.enc.parameters())/1e6))
del m

## 3. Stage 1 — Pretrain tự giám sát (không cần nhãn)
Dùng **toàn bộ** video (train+val+test không nhãn cũng được vì chỉ tái tạo pixel; nhưng để sạch, mặc định chỉ dùng các video không thuộc tập `test`).
Với ~19.900 video nên chạy **100–300 epoch** (A100: khoảng vài giờ/100 epoch). Có checkpoint + resume.

In [ ]:
PRE_EPOCHS, PRE_BS, PRE_LR = 150, 48, 1.5e-4*48/256*4
ids = [i for i in vid_path if split_of(i) != 'test']
dl = torch.utils.data.DataLoader(PretrainDS(ids), batch_size=PRE_BS, shuffle=True, num_workers=8,
                                 drop_last=True, pin_memory=True, persistent_workers=True)
mae = VideoMAE().to(dev)
opt = torch.optim.AdamW(mae.parameters(), lr=PRE_LR, betas=(0.9,0.95), weight_decay=0.05)
total = PRE_EPOCHS*len(dl); warm = 5*len(dl)
lr_at = lambda s: PRE_LR*s/warm if s < warm else PRE_LR*0.5*(1+math.cos(math.pi*(s-warm)/(total-warm)))
ck = f'{CKPT_DIR}/mae_last.pt'; ep0 = 0
if os.path.exists(ck):
    c = torch.load(ck, map_location=dev); mae.load_state_dict(c['m']); opt.load_state_dict(c['o']); ep0 = c['ep']+1
    print('resume từ epoch', ep0)
cmae = torch.compile(mae)
for ep in range(ep0, PRE_EPOCHS):
    t0 = time.time(); tot = 0
    for it, x in enumerate(dl):
        for g in opt.param_groups: g['lr'] = lr_at(ep*len(dl)+it)
        with torch.autocast('cuda', dtype=torch.bfloat16): loss = cmae(x.to(dev, non_blocking=True))
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(mae.parameters(), 1.0); opt.step(); tot += loss.item()
    print(f'ep {ep} loss {tot/len(dl):.4f} {time.time()-t0:.0f}s', flush=True)
    torch.save({'m': mae.state_dict(), 'o': opt.state_dict(), 'ep': ep}, ck)
    if (ep+1) % 25 == 0: torch.save({'enc': mae.enc.state_dict()}, f'{CKPT_DIR}/enc_ep{ep+1}.pt')
torch.save({'enc': mae.enc.state_dict()}, f'{CKPT_DIR}/enc_final.pt')

## 4. Stage 2 — Captioning (dùng encoder do CHÍNH BẠN pretrain ở Stage 1)
Vocab và tokenizer tự xây từ chú thích của bạn (không tải tokenizer ngoài).

In [ ]:
ann = json.load(open(ANN_PATH))
ann = {k: v for k, v in ann.items() if k in vid_path}
print('video có chú thích:', len(ann), '| tổng câu:', sum(len(v['segments']) for v in ann.values()))

tok = lambda s: re.findall(r"[\w']+", s.lower())
cnt = Counter()
for k, v in ann.items():
    if split_of(k) == 'train':
        for sg in v['segments']: cnt.update(tok(sg['caption']))
itos = ['<pad>', '<bos>', '<eos>', '<unk>'] + [w for w, c in cnt.most_common() if c >= 2][:8000]
stoi = {w: i for i, w in enumerate(itos)}; V = len(itos); MAXL = 48
print('vocab:', V)
enc_ids = lambda s: [1] + [stoi.get(w, 3) for w in tok(s)][:MAXL-2] + [2]
json.dump(itos, open(f'{CKPT_DIR}/vocab.json', 'w'))

def seg_items(split):
    return [(k, sg['start'], sg['end'], sg['caption']) for k, v in ann.items() if split_of(k) == split for sg in v['segments']]

def seg_frames(path, st, en, train):
    vr = VideoReader(path, ctx=cpu(0), num_threads=1); n = len(vr); del vr
    a, b = st*FPS, max(en*FPS, st*FPS+T)
    if train:    # jitter nhẹ biên thời gian
        j = 0.1*(b-a); a += random.uniform(-j, j); b += random.uniform(-j, j)
    idx = np.linspace(a, b, T)
    fr, _ = read_frames(path, idx)
    return to_tensor(fr, train)

class CapDS(torch.utils.data.Dataset):
    def __init__(s, items, train): s.items, s.train = items, train
    def __len__(s): return len(s.items)
    def __getitem__(s, k):
        vid, st, en, cap = s.items[k]
        x = seg_frames(vid_path[vid], st, en, s.train)
        ids = enc_ids(cap); ids += [0]*(MAXL-len(ids))
        return x, torch.tensor(ids)

class Captioner(nn.Module):
    def __init__(s, D=384, heads=6, depth=4):
        super().__init__()
        s.enc = Encoder(D)
        s.emb = nn.Embedding(V, D); s.pos = nn.Parameter(torch.zeros(1, MAXL, D)); nn.init.trunc_normal_(s.pos, std=.02)
        layer = nn.TransformerDecoderLayer(D, heads, 4*D, 0.1, activation='gelu', batch_first=True, norm_first=True)
        s.dec = nn.TransformerDecoder(layer, depth); s.norm = nn.LayerNorm(D); s.out = nn.Linear(D, V)
    def decode(s, mem, ids):
        L = ids.shape[1]
        causal = torch.triu(torch.ones(L, L, device=ids.device, dtype=torch.bool), 1)
        h = s.emb(ids) + s.pos[:, :L]
        return s.out(s.norm(s.dec(h, mem, tgt_mask=causal)))
    def forward(s, x, ids): return s.decode(s.enc(x), ids[:, :-1])
    @torch.no_grad()
    def generate(s, x, maxlen=MAXL):
        mem = s.enc(x); B = x.shape[0]
        ids = torch.ones(B, 1, dtype=torch.long, device=x.device)
        for _ in range(maxlen-1):
            nxt = s.decode(mem, ids)[:, -1].argmax(-1, keepdim=True)
            ids = torch.cat([ids, nxt], 1)
            if (ids == 2).any(1).all(): break
        outs = []
        for r in ids.tolist():
            w = []
            for t in r[1:]:
                if t == 2: break
                w.append(itos[t])
            outs.append(' '.join(w))
        return outs

In [ ]:
CAP_EPOCHS, CAP_BS = 40, 32
tr = DataLoaderTr = torch.utils.data.DataLoader(CapDS(seg_items('train'), True), batch_size=CAP_BS, shuffle=True,
        num_workers=8, drop_last=True, pin_memory=True, persistent_workers=True)
va = torch.utils.data.DataLoader(CapDS(seg_items('val'), False), batch_size=CAP_BS, num_workers=4)
cap = Captioner().to(dev)
cap.enc.load_state_dict(torch.load(f'{CKPT_DIR}/enc_final.pt', map_location=dev)['enc'])   # encoder TỰ pretrain

# lr thấp cho encoder, cao cho decoder
enc_p = list(cap.enc.parameters()); other = [p for n, p in cap.named_parameters() if not n.startswith('enc.')]
opt = torch.optim.AdamW([{'params': enc_p, 'lr': 5e-5}, {'params': other, 'lr': 3e-4}], weight_decay=0.05)
base = [g['lr'] for g in opt.param_groups]
total = CAP_EPOCHS*len(tr); warm = len(tr)
ck = f'{CKPT_DIR}/cap_last.pt'; ep0 = 0; best = 1e9
if os.path.exists(ck):
    c = torch.load(ck, map_location=dev); cap.load_state_dict(c['m']); opt.load_state_dict(c['o']); ep0 = c['ep']+1; best = c['best']
for ep in range(ep0, CAP_EPOCHS):
    cap.train(); tot = 0
    for it, (x, ids) in enumerate(tr):
        s = ep*len(tr)+it; f = s/warm if s < warm else 0.5*(1+math.cos(math.pi*(s-warm)/(total-warm)))
        for g, b0 in zip(opt.param_groups, base): g['lr'] = b0*f
        x, ids = x.to(dev), ids.to(dev)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            lg = cap(x, ids)
        loss = F.cross_entropy(lg.float().reshape(-1, V), ids[:, 1:].reshape(-1), ignore_index=0, label_smoothing=0.1)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(cap.parameters(), 1.0); opt.step(); tot += loss.item()
    cap.eval(); vl = 0; n = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        for x, ids in va:
            x, ids = x.to(dev), ids.to(dev)
            vl += F.cross_entropy(cap(x, ids).float().reshape(-1, V), ids[:, 1:].reshape(-1), ignore_index=0).item(); n += 1
    vl /= max(n, 1)
    print(f'ep {ep} train {tot/len(tr):.3f} val {vl:.3f}', flush=True)
    if vl < best: best = vl; torch.save({'m': cap.state_dict()}, f'{CKPT_DIR}/cap_best.pt')
    torch.save({'m': cap.state_dict(), 'o': opt.state_dict(), 'ep': ep, 'best': best}, ck)

## 5. Đánh giá trên tập test (chạy 1 lần duy nhất ở cuối)
Cài thêm metric: `pip install pycocoevalcap` (BLEU/CIDEr, chỉ là phép đo, không phải model).

In [ ]:
!pip -q install pycocoevalcap
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
cap.load_state_dict(torch.load(f'{CKPT_DIR}/cap_best.pt', map_location=dev)['m']); cap.eval()
te_items = seg_items('test')
te = torch.utils.data.DataLoader(CapDS(te_items, False), batch_size=32, num_workers=4)
hyp, ref, i = {}, {}, 0
with torch.autocast('cuda', dtype=torch.bfloat16):
    for x, _ in te:
        for o in cap.generate(x.to(dev)):
            hyp[i] = [o if o else 'a']; ref[i] = [' '.join(tok(te_items[i][3]))]; i += 1
print('BLEU:', Bleu(4).compute_score(ref, hyp)[0]); print('CIDEr:', Cider().compute_score(ref, hyp)[0])
for k in range(5): print('REF:', ref[k][0], '\nHYP:', hyp[k][0], '\n')

## 6. Suy luận: trượt cửa sổ → (start, end, caption)

In [ ]:
@torch.no_grad()
def describe(video_path, windows=(8, 16, 32), stride_ratio=0.5, sim_thr=0.8):
    vr = VideoReader(video_path, ctx=cpu(0), num_threads=1); dur = len(vr)/FPS; del vr
    segs = []
    for w in windows:
        st = 0.0
        while st < max(dur - w*0.5, 1e-6):
            en = min(st + w, dur); segs.append((st, en)); st += w*stride_ratio
    xs = torch.stack([seg_frames(video_path, a, b, False) for a, b in segs]).to(dev)
    caps = []
    with torch.autocast('cuda', dtype=torch.bfloat16):
        for i in range(0, len(xs), 16): caps += cap.generate(xs[i:i+16])
    out, seen = [], []
    for (a, b), c in zip(segs, caps):
        ws = set(c.split())
        if any(len(ws & s)/max(len(ws | s), 1) > sim_thr for s in seen): continue   # bỏ câu trùng
        seen.append(ws); out.append({'start': round(a, 1), 'end': round(b, 1), 'caption': c})
    return sorted(out, key=lambda d: d['start'])

for r in describe(vids[0]): print(r)

> **Giới hạn cần biết:** bước 6 là trượt cửa sổ cố định (chưa có bộ đề xuất đoạn học được). Khi pipeline này chạy ổn, bước nâng cấp tiếp theo là thêm đầu dự đoán biên (start/end) kiểu DETR/PDVC, vẫn khởi tạo từ đầu, huấn luyện trên chính chú thích timestamp của bạn.